# Outlier Detection Strategy for Repetitive Tickets

To detect "outlier" tickets—specifically those indicating an asset is constantly failing despite repairs—we can move beyond simple rule-based filters (like the 90-day logic) to a statistical or machine learning-based approach. Here is the high-level plan:

## 1. Feature Engineering (Creating the Data)
We need to aggregate data at the **Asset** or **Location** level. The model will judge *assets*, not individual tickets, to see if their failure behavior is anomalous.

Key features to calculate for each asset:
*   **Recurrence Count**: Number of corrective tickets in the last X days.
*   **Mean Time Between Failures (MTBF)**: Average days between consecutive tickets. (Lower is worse).
*   **Total Cost/Effort**: Sum of `ACTUAL_COST` or `ACTUAL_HOURS`.
*   **Description Similarity**: (Advanced) How similar are the descriptions? If they are 100% similar, it's definitely the same issue.

## 2. Model Selection
Since we don't have a labeled dataset saying "this asset is definitely broken" (unsupervised learning), we can use:

*   **Isolation Forest**: An algorithm specifically designed to detect anomalies. It works by isolating observations; anomalies are easier to isolate (fewer splits) than normal points. It's effective for high-dimensional data (e.g., combining Frequency + Cost + MTBF).
*   **DBSCAN (Density-Based Clustering)**: Groups "normal" assets together and marks points in low-density regions as outliers.
*   **Statistical Thresholds (Z-Score / IQR)**: Simple but effective. Any asset with a recurrence count > 3 standard deviations from the mean is an outlier.

## 3. Implementation Plan
1.  **Aggregate**: Group `df_tickets` (or the merged asset dataframe) by `ASSET_ID`.
2.  **Calculate Features**: Compute `Ticket_Count`, `Avg_Days_Diff`, `Total_Cost` for each asset.
3.  **Train Model**: Feed these features into an **Isolation Forest**.
4.  **Evaluate**: The model will assign an "anomaly score". Assets with high scores are the ones getting repaired suspiciously often.

We can start by building the dataset for the model.

In [ ]:
from sklearn.ensemble import IsolationForest
import matplotlib.pyplot as plt
import seaborn as sns

# --- 3. Initialize and Train Isolation Forest ---
# contamination: The expected proportion of outliers in the data set.
# We estimate that about 5% of these assets are truly anomalous 'lemons'.
model = IsolationForest(contamination=0.05, random_state=42)

# Select features for the model
features_to_use = ['Ticket_Count', 'Avg_Days_Between_Failures']

# Fit the model
model.fit(df_features[features_to_use])

# --- 4. Predict Anomalies ---
# The model returns -1 for outliers and 1 for inliers (normal)
df_features['anomaly_label'] = model.predict(df_features[features_to_use])

# Calculate Anomaly Score (Lower scores mean more anomalous)
df_features['anomaly_score'] = model.decision_function(df_features[features_to_use])

# Separate outliers for inspection
outliers = df_features[df_features['anomaly_label'] == -1].sort_values(by='anomaly_score')

print(f"Identified {len(outliers)} outliers out of {len(df_features)} assets.")

print("\n--- Top 10 Anomalous Assets (High frequency, low interval) ---")
display(outliers[['ASSET_ID', 'ASSET_NAME', 'Ticket_Count', 'Avg_Days_Between_Failures', 'anomaly_score']].head(10))

# --- 5. Visualize the Results ---
plt.figure(figsize=(12, 8))

# Plot normal points
plt.scatter(df_features[df_features['anomaly_label'] == 1]['Avg_Days_Between_Failures'],
            df_features[df_features['anomaly_label'] == 1]['Ticket_Count'],
            c='blue', label='Normal Asset', alpha=0.5)

# Plot outliers
plt.scatter(outliers['Avg_Days_Between_Failures'],
            outliers['Ticket_Count'],
            c='red', label='Outlier Asset', edgecolor='k', s=100)

plt.title('Isolation Forest Outlier Detection: Asset Failures')
plt.xlabel('Average Days Between Failures (Lower is worse)')
plt.ylabel('Total Ticket Count (Higher is worse)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()